In [11]:
from google.colab import drive
drive.mount('COLAB_DRIVE')

Drive already mounted at COLAB_DRIVE; to attempt to forcibly remount, call drive.mount("COLAB_DRIVE", force_remount=True).


In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tiktoken

import numpy as np

from beartype import beartype
import json
import glob

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

project_name = "nanoGPT" # variable depends on project

if is_colab():
    matches = glob.glob(f"COLAB_DRIVE/MyDrive/**/{project_name}/", recursive=True)
    print(matches)
    if len(matches) == 1:
        proj_dir = matches[0]
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)
    config['seed_settings']['use_seed'] = False

else:
    from root_path import get_root_path
    proj_dir = get_root_path()
    print("Is local")
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)

import sys
sys.path.append(proj_dir)

from utils import torch_ckpt
from model import model

['COLAB_DRIVE/MyDrive/Project/nanoGPT/']


In [13]:
file_path = os.path.join(proj_dir, "Trainer", "Trainer.ipynb")

config['path_settings']['file_path'] = file_path
config['git_settings']['strict_git'] = False

device = config['env_settings']['device']
max_iters = config['deep_learning_settings']['trainer_config']['max_iters']

## Seed Setting

In [14]:
seed_test = torch_ckpt.ckpt_manager(proj_dir, **config)

use_deterministic == True
⚠️  Warning: [Git] Uncommitted changes detected! Commit them for full reproducibility.


## Data Preparation

In [15]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 'COLAB_DRIVE/MyDrive/Project/nanoGPT/',
 '/tmp/tmpqdq12usn',
 'COLAB_DRIVE/MyDrive/Project/nanoGPT/']

In [16]:
from data.data_loader import TokenDataLoader

data_loader = TokenDataLoader(data_train_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "train_all.bin"),
                              data_val_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "val_all_10_to_11.bin"),
                              token_len=config['deep_learning_settings']['model_config']['token_len'],
                              batch_size=config['deep_learning_settings']['data_config']['batch_size'])



## Model definition

In [17]:
if "ff_dim" in config['deep_learning_settings']['model_config'].keys():
    config['deep_learning_settings']['model_config'].pop('ff_dim', None)

In [18]:
gpt_model = model.GPT(**config['deep_learning_settings']['model_config'])

## Optimizer

In [19]:
criterion = nn.CrossEntropyLoss()  # 예: 분류 문제일 때
optimizer = optim.Adam(gpt_model.parameters(), lr=config['deep_learning_settings']['optimizer_config']['lr'])

# Update ckpt_manager with model and optimizer for checkpoint saving
seed_test.deep_learning_settings.model = gpt_model
seed_test.deep_learning_settings.optimizer = optimizer
seed_test.deep_learning_settings.scheduler = None  # No scheduler used
seed_test.deep_learning_settings.grad_scaler = None  # No AMP/mixed precision used

## Data Batch Module 화

In [ ]:
from pathlib import Path
import random
import time
import torch
import numpy as np

print("="*50)
print("trainer start:")
print("device:", device)
print("="*50)

def format_elapsed(seconds):
    minutes = int(seconds // 60)
    secs = seconds - minutes * 60
    return f"{minutes}m {secs:05.2f}s"

train_loss_list = {}
val_loss_list = {}

best_val_loss = float('inf')
patience_counter = 0
max_patience = config['deep_learning_settings']['trainer_config']['patience']

start_time = time.time()

gpt_model.to(device)

for _idx in range(max_iters):
    step_start = time.time()
    print("="*10)
    print(f"Iters: {_idx+1}/{max_iters}")
    x,y = data_loader.get_train_batch()
    x,y = x.to(device), y.to(device)
    pred = gpt_model(x)
    B, T, C = pred.size()
    train_loss = criterion(pred.view(B*T, C), y.view(B*T))
    train_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Store every iteration's loss and batch_size
    train_loss_list[_idx] = {"loss": train_loss.item(), "batch_size": B}

    with torch.no_grad():
        x,y = data_loader.get_val_batch()
        x,y = x.to(device), y.to(device)
        gpt_model.to(device)
        pred = gpt_model(x)
        B, T, C = pred.size()
        val_loss = criterion(pred.view(B*T, C), y.view(B*T))

    # Store every iteration's loss and batch_size
    val_loss_list[_idx] = {"loss": val_loss.item(), "batch_size": B}

    log_lines = []
    should_stop = False

    if _idx % 1 == 0:
        # Compute average for display from recent iterations
        iter_eval = config['deep_learning_settings']['trainer_config']['iter_eval']
        start_idx = max(0, _idx - iter_eval + 1)

        recent_train = [train_loss_list[i] for i in range(start_idx, _idx + 1) if i in train_loss_list]
        train_loss_avg = sum(d["loss"] * d["batch_size"] for d in recent_train) / sum(d["batch_size"] for d in recent_train)
        # log_lines.append(f"{_idx}th iter train loss:{train_loss_avg:.3f}")  # historical average print disabled

        recent_val = [val_loss_list[i] for i in range(start_idx, _idx + 1) if i in val_loss_list]
        val_loss_avg = sum(d["loss"] * d["batch_size"] for d in recent_val) / sum(d["batch_size"] for d in recent_val)
        # log_lines.append(f"{_idx}th iter val loss:{val_loss_avg:.3f}")  # historical average print disabled

        # Update best val loss and patience counter
        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            patience_counter = 0
            # Save checkpoint on new best val loss
            log_lines.append(f"New best val loss: {best_val_loss:.3f} — saving checkpoint at step {_idx + 1}...")
            # seed_test.save_ckpt(
            #     step=_idx + 1,
            #     train_loss_history=train_loss_list,
            #     val_loss_history=val_loss_list,
            #     best_val_loss=best_val_loss,
            #     patience_counter=patience_counter
            # )
        else:
            patience_counter += 1

        # Early stopping check
        if patience_counter >= 200:
            log_lines.append(f"Early stopping at iter {_idx}: val loss has not improved for {max_patience} evaluations")
            should_stop = True

    now = time.time()
    step_time = now - step_start
    elapsed_time = now - start_time
    time_suffix = f" | step {format_elapsed(step_time)} | elapsed {format_elapsed(elapsed_time)}"

    print(f"iter {_idx}: train loss {train_loss.item():.4f}, val loss {val_loss.item():.4f}" + time_suffix)

    for line in log_lines:
        print(line + time_suffix)

    if should_stop:
        break

    # Save checkpoint every 100 steps with worktree backup
    if (_idx + 1) % 100 == 0:
        print(f"Saving checkpoint at step {_idx + 1}..." + time_suffix)
        seed_test.save_ckpt(
            step=_idx + 1,
            train_loss_history=train_loss_list,
            val_loss_history=val_loss_list,
            best_val_loss=best_val_loss,
            patience_counter=patience_counter
        )
        print(f"Checkpoint saved to: {seed_test.session_dir}" + time_suffix)

# Save final checkpoint
print(f"Saving final checkpoint at step {_idx + 1}..." + time_suffix)
seed_test.save_ckpt(
    step=_idx + 1,
    train_loss_history=train_loss_list,
    val_loss_history=val_loss_list,
    best_val_loss=best_val_loss,
    patience_counter=patience_counter
)
print(f"Final checkpoint saved to: {seed_test.session_dir}" + time_suffix)


trainer start:
device: cuda
Iters: 1/100000000
iter 0: train loss 11.3677, val loss 9.9716 | step 0m 00.88s | elapsed 0m 01.01s
0th iter train loss:11.368 | step 0m 00.88s | elapsed 0m 01.01s
0th iter val loss:9.972 | step 0m 00.88s | elapsed 0m 01.01s
New best val loss: 9.972 — saving checkpoint at step 1... | step 0m 00.88s | elapsed 0m 01.01s
Iters: 2/100000000
iter 1: train loss 10.1608, val loss 13.6878 | step 0m 00.83s | elapsed 0m 01.84s
1th iter train loss:10.764 | step 0m 00.83s | elapsed 0m 01.84s
1th iter val loss:11.830 | step 0m 00.83s | elapsed 0m 01.84s
Iters: 3/100000000
iter 2: train loss 13.6391, val loss 10.0371 | step 0m 00.83s | elapsed 0m 02.67s
2th iter train loss:11.723 | step 0m 00.83s | elapsed 0m 02.67s
2th iter val loss:11.232 | step 0m 00.83s | elapsed 0m 02.67s
Iters: 4/100000000
iter 3: train loss 10.3312, val loss 9.4339 | step 0m 00.81s | elapsed 0m 03.48s
3th iter train loss:11.375 | step 0m 00.81s | elapsed 0m 03.48s
3th iter val loss:10.783 | step 0m

### 2025-12-29에 실행한 결과, 5분에 100 iter이다.
### - 500 iter까지 실행한 결과, 약 15분 걸렸다.
